In [28]:
import sys, pandas as pd
print(sys.executable)
print(pd.__version__)

/Users/mehalgosalia/anaconda3/envs/gibleed/bin/python
3.1.0


In [29]:
import os
directory = 'GiBleed_5.5'
print(sorted(os.listdir(directory)))

['CARE_SITE.csv', 'CDM_SOURCE.csv', 'COHORT.csv', 'COHORT_DEFINITION.csv', 'CONCEPT.csv', 'CONCEPT_ANCESTOR.csv', 'CONCEPT_CLASS.csv', 'CONCEPT_METADATA.csv', 'CONCEPT_RELATIONSHIP.csv', 'CONCEPT_RELATIONSHIP_METADATA.csv', 'CONCEPT_SYNONYM.csv', 'CONDITION_ERA.csv', 'CONDITION_OCCURRENCE.csv', 'COST.csv', 'DEATH.csv', 'DEVICE_EXPOSURE.csv', 'DOMAIN.csv', 'DOSE_ERA.csv', 'DRUG_ERA.csv', 'DRUG_EXPOSURE.csv', 'DRUG_STRENGTH.csv', 'EPISODE.csv', 'EPISODE_EVENT.csv', 'FACT_RELATIONSHIP.csv', 'LOCATION.csv', 'MEASUREMENT.csv', 'METADATA.csv', 'NOTE.csv', 'NOTE_NLP.csv', 'OBSERVATION.csv', 'OBSERVATION_PERIOD.csv', 'PACK_CONTENT.csv', 'PAYER_PLAN_PERIOD.csv', 'PERSON.csv', 'PROCEDURE_OCCURRENCE.csv', 'PROVIDER.csv', 'RELATIONSHIP.csv', 'SOURCE_TO_CONCEPT_MAP.csv', 'SPECIMEN.csv', 'VISIT_DETAIL.csv', 'VISIT_OCCURRENCE.csv', 'VOCABULARY.csv']


In [30]:
table_names = ['PERSON', 'OBSERVATION_PERIOD', 'DRUG_EXPOSURE',
               'CONDITION_OCCURRENCE', 'VISIT_OCCURRENCE', 'CONCEPT', 'CONCEPT_ANCESTOR']
tables = {}

for name in table_names:
    path = os.path.join(directory, name + ".csv")   # build the file name from `name`
    tables[name] = pd.read_csv(path)       # read the file at `path`
    print(name, len(tables[name]), list(tables[name].columns))              # number of rows, list of columns

PERSON 2694 ['PERSON_ID', 'GENDER_CONCEPT_ID', 'YEAR_OF_BIRTH', 'MONTH_OF_BIRTH', 'DAY_OF_BIRTH', 'BIRTH_DATETIME', 'RACE_CONCEPT_ID', 'ETHNICITY_CONCEPT_ID', 'LOCATION_ID', 'PROVIDER_ID', 'CARE_SITE_ID', 'PERSON_SOURCE_VALUE', 'GENDER_SOURCE_VALUE', 'GENDER_SOURCE_CONCEPT_ID', 'RACE_SOURCE_VALUE', 'RACE_SOURCE_CONCEPT_ID', 'ETHNICITY_SOURCE_VALUE', 'ETHNICITY_SOURCE_CONCEPT_ID']
OBSERVATION_PERIOD 5343 ['OBSERVATION_PERIOD_ID', 'PERSON_ID', 'OBSERVATION_PERIOD_START_DATE', 'OBSERVATION_PERIOD_END_DATE', 'PERIOD_TYPE_CONCEPT_ID']
DRUG_EXPOSURE 67707 ['DRUG_EXPOSURE_ID', 'PERSON_ID', 'DRUG_CONCEPT_ID', 'DRUG_EXPOSURE_START_DATE', 'DRUG_EXPOSURE_START_DATETIME', 'DRUG_EXPOSURE_END_DATE', 'DRUG_EXPOSURE_END_DATETIME', 'VERBATIM_END_DATE', 'DRUG_TYPE_CONCEPT_ID', 'STOP_REASON', 'REFILLS', 'QUANTITY', 'DAYS_SUPPLY', 'SIG', 'ROUTE_CONCEPT_ID', 'LOT_NUMBER', 'PROVIDER_ID', 'VISIT_OCCURRENCE_ID', 'VISIT_DETAIL_ID', 'DRUG_SOURCE_VALUE', 'DRUG_SOURCE_CONCEPT_ID', 'ROUTE_SOURCE_VALUE', 'DOSE_UNIT

In [31]:
import duckdb
person = tables["PERSON"]
obs_period = tables["OBSERVATION_PERIOD"]
duckdb.sql("SELECT COUNT (DISTINCT PERSON_ID) FROM person")
duckdb.sql("""
    SELECT n, COUNT(*) AS n_people
    FROM (
        SELECT PERSON_ID, COUNT(*) AS n
        FROM obs_period
        GROUP BY PERSON_ID
    ) AS t
    GROUP BY n
""")

nBIGINT,n_peopleBIGINT
1,5343


In [32]:
duckdb.sql("""
    SELECT
        CASE
            WHEN p.person_id IS NULL THEN 'obs_period_only'
            WHEN o.person_id IS NULL THEN 'person_only'
            ELSE 'both'
        END AS membership,
        COUNT(*) AS n
    FROM (SELECT DISTINCT person_id FROM person) AS p
    FULL OUTER JOIN
         (SELECT DISTINCT person_id FROM obs_period) AS o
      ON p.person_id = o.person_id
    GROUP BY membership
    ORDER BY membership
""")

membershipVARCHAR,nBIGINT
both,2694
obs_period_only,2649


In [33]:
drug_exp = tables["DRUG_EXPOSURE"]
cond_occ = tables["CONDITION_OCCURRENCE"]
duckdb.sql("""
    SELECT COUNT(DISTINCT person_id) AS n
    FROM drug_exp                                        
    WHERE person_id NOT IN (SELECT person_id FROM person)   
""")


nBIGINT
0


In [34]:
duckdb.sql("""
    SELECT COUNT(DISTINCT person_id) AS n
    FROM cond_occ                                       
    WHERE person_id NOT IN (SELECT person_id FROM person)   
""")

nBIGINT
0


## Step 2 — Concept sets
Find every standard concept for the exposures and the outcome using CONCEPT_ANCESTOR (ingredient/finding → all descendants), then check which of them actually appear in the patient data.

In [35]:
concept = tables["CONCEPT"]
concept_anc = tables["CONCEPT_ANCESTOR"]
concept_rel = pd.read_csv(os.path.join(directory, "CONCEPT_RELATIONSHIP.csv"))

In [36]:
# exposure concept set: all descendants of the two ingredients
drug_cs = duckdb.sql("""
    SELECT ca.ancestor_concept_id AS ingredient_id,
           c.concept_id, c.concept_name, c.concept_class_id, c.standard_concept
    FROM concept_anc AS ca
    JOIN concept AS c ON c.concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id IN (1118084, 1124300)
    ORDER BY ingredient_id, c.concept_id
""").df()
drug_cs

,ingredient_id,CONCEPT_ID,CONCEPT_NAME,CONCEPT_CLASS_ID,STANDARD_CONCEPT
0,1118084,1118084,celecoxib,Ingredient,S
1,1124300,1124300,Diclofenac,Ingredient,S


In [37]:
# outcome concept set: all descendants of Gastrointestinal hemorrhage
gi_cs = duckdb.sql("""
    SELECT ca.ancestor_concept_id AS parent_id,
           c.concept_id, c.concept_name, c.concept_class_id, c.standard_concept
    FROM concept_anc AS ca
    JOIN concept AS c ON c.concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id = 192671
    ORDER BY c.concept_id
""").df()
gi_cs

,parent_id,CONCEPT_ID,CONCEPT_NAME,CONCEPT_CLASS_ID,STANDARD_CONCEPT
0,192671,192671,Gastrointestinal hemorrhage,Clinical Finding,S


In [38]:
# coverage: which drug concepts are used in DRUG_EXPOSURE
duckdb.sql("""
    SELECT cs.ingredient_id, cs.concept_name, COUNT(*) AS n_rows,
           COUNT(DISTINCT d.person_id) AS n_people
    FROM drug_cs AS cs
    JOIN drug_exp AS d ON d.drug_concept_id = cs.concept_id
    GROUP BY ALL
    ORDER BY n_rows DESC
""")

ingredient_idBIGINT,CONCEPT_NAMEVARCHAR,n_rowsBIGINT,n_peopleBIGINT
1118084,celecoxib,1844,1844
1124300,Diclofenac,850,850


In [39]:
# coverage: which GI bleed concepts are used in CONDITION_OCCURRENCE
duckdb.sql("""
    SELECT cs.concept_name, COUNT(*) AS n_rows,
           COUNT(DISTINCT c.person_id) AS n_people
    FROM gi_cs AS cs
    JOIN cond_occ AS c ON c.condition_concept_id = cs.concept_id
    GROUP BY ALL
    ORDER BY n_rows DESC
""")

CONCEPT_NAMEVARCHAR,n_rowsBIGINT,n_peopleBIGINT
Gastrointestinal hemorrhage,479,479


In [40]:
# incomplete vocabulary: example formulations exist in CONCEPT but have no ancestor link to their ingredient
duckdb.sql("""
    SELECT c.concept_id, c.concept_name, c.concept_class_id,
           EXISTS (SELECT 1 FROM concept_anc AS ca
                   WHERE ca.descendant_concept_id = c.concept_id
                     AND ca.ancestor_concept_id IN (1118084, 1124300)) AS linked_to_ingredient
    FROM concept AS c
    WHERE c.concept_id IN (1118088, 40162359)
""")

CONCEPT_IDBIGINT,CONCEPT_NAMEVARCHAR,CONCEPT_CLASS_IDVARCHAR,linked_to_ingredientBOOLEAN
1118088,celecoxib 200 MG Oral Capsule [Celebrex],Branded Drug,False
40162359,Diclofenac Sodium 75 MG Delayed Release Oral Tablet,Clinical Drug,False


In [41]:
# source mapping (for understanding only): the ICD10CM code is non-standard and maps to SNOMED 192671
display(concept[concept["CONCEPT_ID"] == 35208414][["CONCEPT_ID", "CONCEPT_NAME", "VOCABULARY_ID", "STANDARD_CONCEPT"]])
display(concept_rel[(concept_rel["CONCEPT_ID_1"] == 35208414) & (concept_rel["RELATIONSHIP_ID"] == "Maps to")])
duckdb.sql("""
    SELECT condition_concept_id, condition_source_concept_id, COUNT(*) AS n_rows
    FROM cond_occ
    WHERE condition_concept_id = 192671
    GROUP BY ALL
""")

,CONCEPT_ID,CONCEPT_NAME,VOCABULARY_ID,STANDARD_CONCEPT
0,35208414,"Gastrointestinal hemorrhage, unspecified",ICD10CM,NaN


,CONCEPT_ID_1,CONCEPT_ID_2,RELATIONSHIP_ID,VALID_START_DATE,VALID_END_DATE,INVALID_REASON
3,35208414,192671,Maps to,1970-01-01,2099-12-31,NaN


CONDITION_CONCEPT_IDBIGINT,CONDITION_SOURCE_CONCEPT_IDBIGINT,n_rowsBIGINT
192671,35208414,479


In [42]:
# records per person per drug, and patients with both drugs
display(duckdb.sql("""
    SELECT drug_concept_id, MAX(n) AS max_rows_per_person
    FROM (SELECT person_id, drug_concept_id, COUNT(*) AS n
          FROM drug_exp
          WHERE drug_concept_id IN (1118084, 1124300)
          GROUP BY ALL)
    GROUP BY ALL
"""))
duckdb.sql("""
    SELECT COUNT(*) AS n_both_drugs
    FROM (SELECT person_id
          FROM drug_exp
          WHERE drug_concept_id IN (1118084, 1124300)
          GROUP BY person_id
          HAVING COUNT(DISTINCT drug_concept_id) = 2)
""")

DRUG_CONCEPT_IDBIGINT,max_rows_per_personBIGINT
1124300,1
1118084,1


n_both_drugsBIGINT
0


In [43]:
# final concept-set table (as required by the brief)
concept_set_table = pd.DataFrame([
    (1118084, "celecoxib", "Exposure: treatment = 1", "Ingredient + all CONCEPT_ANCESTOR descendants via DRUG_CONCEPT_ID"),
    (1124300, "Diclofenac", "Exposure: comparator = 0", "Ingredient + all CONCEPT_ANCESTOR descendants via DRUG_CONCEPT_ID"),
    (192671, "Gastrointestinal hemorrhage", "Outcome / baseline exclusion", "Standard SNOMED concept + descendants via CONDITION_CONCEPT_ID"),
    (35208414, "Gastrointestinal hemorrhage, unspecified (ICD10CM)", "Source code: mapping check only", "Not used for selection; maps to 192671"),
], columns=["concept_id", "concept_name", "role", "inclusion_rule"])
concept_set_table

,concept_id,concept_name,role,inclusion_rule
0,1118084,celecoxib,Exposure: treatment = 1,Ingredient + all CONCEPT_ANCESTOR descendants ...
1,1124300,Diclofenac,Exposure: comparator = 0,Ingredient + all CONCEPT_ANCESTOR descendants ...
2,192671,Gastrointestinal hemorrhage,Outcome / baseline exclusion,Standard SNOMED concept + descendants via COND...
3,35208414,"Gastrointestinal hemorrhage, unspecified (ICD1...",Source code: mapping check only,Not used for selection; maps to 192671


## Step 3 — Cohort construction and attrition
Apply the protocol's cohort rules one at a time and count the patients remaining after each step.

In [45]:
# rules 1–2: index date = earliest celecoxib/diclofenac start per patient; treatment 1 = celecoxib, 0 = diclofenac
index_df = duckdb.sql("""
    SELECT person_id,
           MIN(CAST(drug_exposure_start_date AS DATE)) AS index_date,
           MAX(CASE WHEN drug_concept_id = 1118084 THEN 1 ELSE 0 END) AS treatment,
           COUNT(DISTINCT drug_concept_id) AS n_drugs
    FROM drug_exp
    WHERE drug_concept_id IN (1118084, 1124300)
    GROUP BY person_id
""").df()

display(index_df.head())
display(index_df.dtypes)
index_df["treatment"].value_counts()

,PERSON_ID,index_date,treatment,n_drugs
0,2230,2009-01-28,1,1
1,4916,2013-01-06,1,1
2,1068,2014-11-23,1,1
3,2346,1986-11-13,1,1
4,3399,2016-08-13,1,1


PERSON_ID              int64
index_date    datetime64[us]
treatment              int32
n_drugs                int64
dtype: object

treatment
1    1844
0     850
Name: count, dtype: int64

In [50]:
# attrition tracker: patients remaining after each rule
attrition = [("Patients with a celecoxib or diclofenac exposure", len(index_df))]

# rule 1: exclude same-day dual starts (both drugs on the earliest date)
cohort = index_df[index_df["n_drugs"] == 1].copy()
attrition.append(("Exclude same-day dual starts", len(cohort)))

# rule 2: new-user washout — no study-drug exposure in index −365 to −1
prior_use = duckdb.sql("""
    SELECT DISTINCT c.person_id
    FROM cohort AS c
    JOIN drug_exp AS d ON d.person_id = c.person_id
    WHERE d.drug_concept_id IN (1118084, 1124300)
      AND CAST(d.drug_exposure_start_date AS DATE)
          BETWEEN c.index_date - INTERVAL 365 DAY AND c.index_date - INTERVAL 1 DAY
""").df()
cohort = cohort[~cohort["PERSON_ID"].isin(prior_use["PERSON_ID"])]
attrition.append(("Exclude prior study-drug use in 365-day baseline (new users)", len(cohort)))

pd.DataFrame(attrition, columns=["step", "n_remaining"])

,step,n_remaining
0,Patients with a celecoxib or diclofenac exposure,2694
1,Exclude same-day dual starts,2694
2,Exclude prior study-drug use in 365-day baseli...,2694


In [51]:
# rule 3: observation period must cover index −365 through index +90 (one period per person, so no gaps)
covered = duckdb.sql("""
    SELECT c.person_id
    FROM cohort AS c
    JOIN obs_period AS o ON o.person_id = c.person_id
    WHERE CAST(o.observation_period_start_date AS DATE) <= c.index_date - INTERVAL 365 DAY
      AND CAST(o.observation_period_end_date   AS DATE) >= c.index_date + INTERVAL 90 DAY
""").df()
cohort = cohort[cohort["PERSON_ID"].isin(covered["PERSON_ID"])]
attrition.append(("Continuous observation index −365 to +90", len(cohort)))

display(pd.DataFrame(attrition, columns=["step", "n_remaining"]))
cohort["treatment"].value_counts()


,step,n_remaining
0,Patients with a celecoxib or diclofenac exposure,2694
1,Exclude same-day dual starts,2694
2,Exclude prior study-drug use in 365-day baseli...,2694
3,Continuous observation index −365 to +90,2605


treatment
1    1781
0     824
Name: count, dtype: int64

In [52]:
# rule 4: exclude patients with a GI bleed (concept 192671) in the 365-day baseline (incident-event analysis)
baseline_gi = duckdb.sql("""
    SELECT DISTINCT c.person_id
    FROM cohort AS c
    JOIN cond_occ AS co ON co.person_id = c.person_id
    WHERE co.condition_concept_id = 192671
      AND CAST(co.condition_start_date AS DATE)
          BETWEEN c.index_date - INTERVAL 365 DAY AND c.index_date - INTERVAL 1 DAY
""").df()
cohort = cohort[~cohort["PERSON_ID"].isin(baseline_gi["PERSON_ID"])]
attrition.append(("Exclude GI bleed in 365-day baseline", len(cohort)))

attrition_df = pd.DataFrame(attrition, columns=["step", "n_remaining"])
attrition_df["n_excluded"] = -attrition_df["n_remaining"].diff()
display(attrition_df)
cohort["treatment"].value_counts()

,step,n_remaining,n_excluded
0,Patients with a celecoxib or diclofenac exposure,2694,NaN
1,Exclude same-day dual starts,2694,-0.0
2,Exclude prior study-drug use in 365-day baseli...,2694,-0.0
3,Continuous observation index −365 to +90,2605,89.0
4,Exclude GI bleed in 365-day baseline,2605,-0.0


treatment
1    1781
0     824
Name: count, dtype: int64

## Step 4 - Outcomes + Covariates -> Analysis dataset

In [53]:
# rules 5–8: outcome, covariates and one-row-per-patient analysis dataset
vis_occ = tables["VISIT_OCCURRENCE"]

ads = duckdb.sql("""
    SELECT c.person_id AS patient_id,
           c.index_date,
           c.treatment,
           -- rule 5: first GI bleed on index +1 to +90 (patient-level flag, not row count)
           CASE WHEN EXISTS (
               SELECT 1 FROM cond_occ AS co
               WHERE co.person_id = c.person_id
                 AND co.condition_concept_id = 192671
                 AND CAST(co.condition_start_date AS DATE)
                     BETWEEN c.index_date + INTERVAL 1 DAY AND c.index_date + INTERVAL 90 DAY
           ) THEN 1 ELSE 0 END AS gi_bleed_90d,
           -- rule 6: exact age at index (full birth date is available)
           date_sub('year', make_date(p.year_of_birth, p.month_of_birth, p.day_of_birth), c.index_date) AS age,
           CASE p.gender_concept_id WHEN 8507 THEN 'M' WHEN 8532 THEN 'F' END AS sex,
           -- rule 6: baseline record counts in index −365 to −1
           (SELECT COUNT(DISTINCT co.condition_occurrence_id) FROM cond_occ AS co
             WHERE co.person_id = c.person_id
               AND CAST(co.condition_start_date AS DATE)
                   BETWEEN c.index_date - INTERVAL 365 DAY AND c.index_date - INTERVAL 1 DAY) AS baseline_condition_count,
           (SELECT COUNT(DISTINCT d.drug_exposure_id) FROM drug_exp AS d
             WHERE d.person_id = c.person_id
               AND CAST(d.drug_exposure_start_date AS DATE)
                   BETWEEN c.index_date - INTERVAL 365 DAY AND c.index_date - INTERVAL 1 DAY) AS baseline_drug_count,
           (SELECT COUNT(DISTINCT v.visit_occurrence_id) FROM vis_occ AS v
             WHERE v.person_id = c.person_id
               AND CAST(v.visit_start_date AS DATE)
                   BETWEEN c.index_date - INTERVAL 365 DAY AND c.index_date - INTERVAL 1 DAY) AS baseline_visit_count
    FROM cohort AS c
    JOIN person AS p ON p.person_id = c.person_id
    ORDER BY patient_id
""").df()

# checks: one row per patient, no missing values
print(len(ads), "rows | unique patients:", ads["patient_id"].is_unique, "| missing values:", ads.isna().sum().sum())
display(ads.head())
ads.groupby("treatment")["gi_bleed_90d"].agg(n="size", events="sum", risk="mean")

2605 rows | unique patients: True | missing values: 0


,patient_id,index_date,treatment,gi_bleed_90d,age,sex,baseline_condition_count,baseline_drug_count,baseline_visit_count
0,1,1982-08-12,1,0,33,M,1,0,1
1,2,1956-12-04,1,0,36,F,0,0,0
2,3,1957-12-08,1,1,41,M,2,2,0
3,5,2009-05-30,1,0,40,M,0,1,0
4,6,2005-07-13,1,0,41,F,1,1,0


,n,events,risk
treatment,,,
0,824,121,0.146845
1,1781,343,0.192588


In [54]:
# rule 7 diagnostic: crossover = initiators of one drug who also have a record of the other within 90 days
crossover = duckdb.sql("""
    SELECT a.treatment, COUNT(DISTINCT a.patient_id) AS n_crossover
    FROM ads AS a
    JOIN drug_exp AS d ON d.person_id = a.patient_id
    WHERE d.drug_concept_id = CASE WHEN a.treatment = 1 THEN 1124300 ELSE 1118084 END
      AND CAST(d.drug_exposure_start_date AS DATE)
          BETWEEN a.index_date + INTERVAL 1 DAY AND a.index_date + INTERVAL 90 DAY
    GROUP BY a.treatment
""").df()
display(crossover)   # empty table = 0 crossovers in both arms

# covariate distributions: how sparse are the baseline counts?
display(ads[["age", "baseline_condition_count", "baseline_drug_count", "baseline_visit_count"]].describe().round(2))
display(ads["baseline_visit_count"].value_counts())

# data dictionary for derived fields (required by the brief)
data_dictionary = pd.DataFrame([
    ("patient_id", "int", "PERSON.PERSON_ID", "One row per patient"),
    ("index_date", "date", "DRUG_EXPOSURE", "Earliest celecoxib or diclofenac DRUG_EXPOSURE_START_DATE"),
    ("treatment", "0/1", "DRUG_EXPOSURE", "1 = celecoxib (1118084), 0 = diclofenac (1124300); fixed at index (ITT)"),
    ("gi_bleed_90d", "0/1", "CONDITION_OCCURRENCE", "1 if any GI hemorrhage (192671) starts on index +1 to +90; patient-level"),
    ("age", "int (years)", "PERSON", "Completed years from full birth date (year/month/day) to index_date"),
    ("sex", "M/F", "PERSON", "GENDER_CONCEPT_ID 8507 = M, 8532 = F"),
    ("baseline_condition_count", "int", "CONDITION_OCCURRENCE", "Distinct condition records starting index −365 to −1 (records, not unique diseases)"),
    ("baseline_drug_count", "int", "DRUG_EXPOSURE", "Distinct drug exposure records starting index −365 to −1 (records, not unique ingredients)"),
    ("baseline_visit_count", "int", "VISIT_OCCURRENCE", "Distinct visit records starting index −365 to −1"),
], columns=["field", "type", "source_table", "definition"])
data_dictionary

,treatment,n_crossover


,age,baseline_condition_count,baseline_drug_count,baseline_visit_count
count,2605.00,2605.00,2605.00,2605.00
mean,38.15,0.57,0.33,0.00
std,3.28,0.79,0.72,0.04
min,31.00,0.00,0.00,0.00
25%,36.00,0.00,0.00,0.00
50%,38.00,0.00,0.00,0.00
75%,41.00,1.00,0.00,0.00
max,46.00,5.00,12.00,1.00


baseline_visit_count
0    2601
1       4
Name: count, dtype: int64

,field,type,source_table,definition
0,patient_id,int,PERSON.PERSON_ID,One row per patient
1,index_date,date,DRUG_EXPOSURE,Earliest celecoxib or diclofenac DRUG_EXPOSURE...
2,treatment,0/1,DRUG_EXPOSURE,"1 = celecoxib (1118084), 0 = diclofenac (11243..."
3,gi_bleed_90d,0/1,CONDITION_OCCURRENCE,1 if any GI hemorrhage (192671) starts on inde...
4,age,int (years),PERSON,Completed years from full birth date (year/mon...
5,sex,M/F,PERSON,"GENDER_CONCEPT_ID 8507 = M, 8532 = F"
6,baseline_condition_count,int,CONDITION_OCCURRENCE,Distinct condition records starting index −365...
7,baseline_drug_count,int,DRUG_EXPOSURE,Distinct drug exposure records starting index ...
8,baseline_visit_count,int,VISIT_OCCURRENCE,Distinct visit records starting index −365 to −1
